# 07 --- SUS score computation

This notebook computes the System Usability Scale (SUS) score for every guardian and the aggregate mean and standard deviation reported as **81.46 ± 11.65** in the manuscript.

**Input:** `../data/raw/sus-responses.csv`

**Output:**
- `../data/processed/sus-scores.csv` (per-participant score 0--100 and qualitative rating)
- `../data/processed/sus-summary.csv` (mean, standard deviation, adjective rating)
- Bar chart with error bars (in-notebook output)

**SUS scoring rule** (Brooke, 1996):

1. For every odd-numbered item (1, 3, 5, 7, 9), subtract 1 from the score.
2. For every even-numbered item (2, 4, 6, 8, 10), subtract the score from 5.
3. Sum the transformed scores of the 10 items (range 0--40) and multiply by 2.5 to obtain the SUS score on a 0--100 scale.

**Adjective ratings** (Bangor et al., 2008): worst imaginable (< 25), poor (25--51), acceptable (52--72), good (73--85), excellent (86--100), best imaginable (= 100).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv('../data/raw/sus-responses.csv', comment='#')
df.head()

In [ ]:
def sus_score(row):
    odd_items = ['sus_1', 'sus_3', 'sus_5', 'sus_7', 'sus_9']
    even_items = ['sus_2', 'sus_4', 'sus_6', 'sus_8', 'sus_10']
    odd = sum(row[i] - 1 for i in odd_items)
    even = sum(5 - row[i] for i in even_items)
    return (odd + even) * 2.5


def adjective_rating(score):
    if score < 25:
        return 'Worst imaginable'
    if score < 52:
        return 'Poor'
    if score < 73:
        return 'Acceptable'
    if score < 86:
        return 'Good'
    if score < 100:
        return 'Excellent'
    return 'Best imaginable'


df['sus_score'] = df.apply(sus_score, axis=1)
df['adjective_rating'] = df['sus_score'].apply(adjective_rating)
df[['participant_id', 'sus_score', 'adjective_rating']].to_csv(
    '../data/processed/sus-scores.csv', index=False
)
df[['participant_id', 'sus_score', 'adjective_rating']]

In [ ]:
summary = pd.DataFrame({
    'n': [len(df)],
    'mean_sus_score': [df['sus_score'].mean()],
    'std_sus_score': [df['sus_score'].std(ddof=1)],
    'aggregate_rating': [adjective_rating(df['sus_score'].mean())],
})
summary.to_csv('../data/processed/sus-summary.csv', index=False)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df['participant_id'], df['sus_score'], color='steelblue', label='Participant')
mean_score = df['sus_score'].mean()
std_score = df['sus_score'].std(ddof=1)
ax.axhline(mean_score, linestyle='--', color='darkorange', label=f'Mean = {mean_score:.2f}')
ax.fill_between(range(-1, len(df) + 1), mean_score - std_score, mean_score + std_score,
                color='darkorange', alpha=0.15, label=f'\u00b11 SD = {std_score:.2f}')
ax.set_xlabel('Participant')
ax.set_ylabel('SUS Score')
ax.set_ylim(0, 100)
ax.set_title('SUS scores per guardian, with mean and standard deviation')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/sus-scores-figure.pdf')
plt.show()